# Clipt — Real-ESRGAN Video Upscaling

Upscales detected game film clips from 360p to 720p using AI super-resolution.

Run on Google Colab A100 GPU (`Runtime > Change runtime type > A100 GPU`).

**Step 1 of 2:** Run this notebook first, then run `clipt_jersey_ocr_training.ipynb`.

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500] if result.returncode == 0 else 'No GPU detected — change runtime to A100')

In [ ]:
# Install Real-ESRGAN and dependencies
# This cell handles the known basicsr/torchvision compatibility issue
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'ERROR: {cmd}\n{result.stderr[-500:]}')
    else:
        print(f'OK: {cmd[:60]}')
    return result.returncode == 0

# Install in correct order to avoid conflicts
run('pip install -q basicsr facexlib gfpgan')
run('pip install -q realesrgan')
run('pip install -q cloudinary yt-dlp')

# Clone Real-ESRGAN repo for video inference script
import os
if not os.path.exists('/content/Real-ESRGAN'):
    run('git clone https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN')

os.chdir('/content/Real-ESRGAN')
run('pip install -q -r requirements.txt')
run('python setup.py develop -q')

# Fix known basicsr compatibility issue with newer torchvision
# (functional_tensor was removed, replaced with functional)
import glob
for fix_path in glob.glob('/usr/local/lib/python*/dist-packages/basicsr/data/degradations.py'):
    with open(fix_path, 'r') as f:
        content = f.read()
    if 'functional_tensor' in content:
        content = content.replace(
            'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
            'from torchvision.transforms.functional import rgb_to_grayscale'
        )
        with open(fix_path, 'w') as f:
            f.write(content)
        print(f'Fixed basicsr compatibility issue at {fix_path}')

# Download model weights
os.makedirs('/content/Real-ESRGAN/weights', exist_ok=True)
if not os.path.exists('/content/Real-ESRGAN/weights/RealESRGAN_x4plus.pth'):
    run('wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P /content/Real-ESRGAN/weights/')
    print('Model weights downloaded')
else:
    print('Model weights already present')

print('\nAll dependencies installed successfully')

In [ ]:
# ============================================================
# EDIT THIS CELL — fill in your credentials
# ============================================================

# Cloudinary credentials (from https://cloudinary.com/console)
CLOUDINARY_CLOUD_NAME = 'dc33vjyyv'  # Already set for Clipt
CLOUDINARY_API_KEY = 'PASTE_YOUR_API_KEY_HERE'
CLOUDINARY_API_SECRET = 'PASTE_YOUR_API_SECRET_HERE'

# The YouTube URL of the game film to process
YOUTUBE_URL = 'https://www.youtube.com/watch?v=x0bZnfVHDx4'

# Clips to upscale — timestamps from detection results
# These are the 3 vision-verified real plays from Dustin's game
CLIPS = [
    {"id": 1, "startTime": 1686, "endTime": 1701, "label": "snap-run-tackle"},
    {"id": 2, "startTime": 5300, "endTime": 5315, "label": "play-action"},
    {"id": 3, "startTime": 4413, "endTime": 4428, "label": "snap-tackle"},
]

# ============================================================
# DO NOT EDIT BELOW THIS LINE
# ============================================================

import cloudinary
cloudinary.config(
    cloud_name=CLOUDINARY_CLOUD_NAME,
    api_key=CLOUDINARY_API_KEY,
    api_secret=CLOUDINARY_API_SECRET,
)
print(f'Cloudinary configured: {CLOUDINARY_CLOUD_NAME}')
print(f'Processing {len(CLIPS)} clips from: {YOUTUBE_URL}')

In [ ]:
import os, subprocess, json

os.makedirs('/content/clips_360p', exist_ok=True)
os.makedirs('/content/clips_720p', exist_ok=True)

source_path = '/content/source.mp4'

print('Downloading game film from YouTube...')
print('This may take 3-5 minutes for a 2-hour game at 360p')

result = subprocess.run([
    'yt-dlp',
    '-f', 'best[height<=480][ext=mp4]/best[height<=480]/best',
    '-o', source_path,
    '--no-playlist',
    YOUTUBE_URL
], capture_output=True, text=True)

if result.returncode != 0:
    print(f'Download failed: {result.stderr[-500:]}')
else:
    size_mb = os.path.getsize(source_path) // 1024 // 1024
    # Probe resolution
    probe = subprocess.run(
        ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_streams', source_path],
        capture_output=True, text=True
    )
    info = json.loads(probe.stdout)
    for s in info.get('streams', []):
        if s.get('codec_type') == 'video':
            print(f'Downloaded: {size_mb}MB at {s["width"]}x{s["height"]}')
    print('Download complete')

In [ ]:
import time, cloudinary.uploader

upscaled_results = []

for clip in CLIPS:
    clip_id = clip['id']
    start = clip['startTime']
    end = clip['endTime']
    label = clip.get('label', f'clip_{clip_id}')

    input_path = f'/content/clips_360p/clip_{clip_id}_{start}s.mp4'
    output_path = f'/content/clips_720p/clip_{clip_id}_{start}s_720p.mp4'

    print(f'\nProcessing clip {clip_id}: {start}s-{end}s ({label})')

    # Step 1: Extract clip segment
    extract_result = subprocess.run([
        'ffmpeg', '-y',
        '-ss', str(start), '-to', str(end),
        '-i', source_path,
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
        '-c:a', 'aac', input_path
    ], capture_output=True, text=True)

    if extract_result.returncode != 0:
        print(f'  Extract failed: {extract_result.stderr[-200:]}')
        continue

    input_size = os.path.getsize(input_path) // 1024
    print(f'  Extracted: {input_size}KB')

    # Step 2: Upscale with Real-ESRGAN
    print(f'  Upscaling with Real-ESRGAN (this takes 30-60s per clip on A100)...')
    t0 = time.time()

    upscale_result = subprocess.run([
        'python', '/content/Real-ESRGAN/inference_realesrgan_video.py',
        '-n', 'RealESRGAN_x4plus',
        '-i', input_path,
        '-o', output_path,
        '--outscale', '2',
        '--fp32'
    ], capture_output=True, text=True, cwd='/content/Real-ESRGAN')

    elapsed = time.time() - t0

    if not os.path.exists(output_path) or os.path.getsize(output_path) < 1000:
        print(f'  Upscale failed ({elapsed:.0f}s): {upscale_result.stderr[-300:]}')
        continue

    # Verify output resolution
    probe = subprocess.run(
        ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_streams', output_path],
        capture_output=True, text=True
    )
    info = json.loads(probe.stdout)
    out_w = out_h = 0
    for s in info.get('streams', []):
        if s.get('codec_type') == 'video':
            out_w, out_h = s['width'], s['height']

    out_size = os.path.getsize(output_path) // 1024 // 1024
    print(f'  Upscaled: {out_w}x{out_h} {out_size}MB in {elapsed:.0f}s')

    # Step 3: Upload to Cloudinary
    print(f'  Uploading to Cloudinary...')
    upload = cloudinary.uploader.upload_large(
        output_path,
        resource_type='video',
        folder='clipt-clips-720p',
        public_id=f'clip_{clip_id}_{start}s_720p',
        overwrite=True,
    )

    cloudinary_url = upload.get('secure_url', '')
    print(f'  Uploaded: {cloudinary_url}')

    upscaled_results.append({
        'clip_id': clip_id,
        'startTime': start,
        'endTime': end,
        'label': label,
        'resolution': f'{out_w}x{out_h}',
        'cloudinary_url': cloudinary_url,
    })

print(f'\n=== UPSCALING COMPLETE ===')
print(f'Successfully upscaled: {len(upscaled_results)}/{len(CLIPS)} clips')

In [ ]:
print('=== COPY THESE URLs INTO clipt_jersey_ocr_training.ipynb ===')
print()
print('CLIP_URLS = [')
for r in upscaled_results:
    print(f'    "{r["cloudinary_url"]}",  # clip {r["clip_id"]} {r["startTime"]}s-{r["endTime"]}s {r["label"]}')
print(']')
print()
print('=== CLIP DETAILS ===')
for r in upscaled_results:
    print(f'Clip {r["clip_id"]}: {r["startTime"]}s-{r["endTime"]}s | {r["resolution"]} | {r["cloudinary_url"]}')